In [ ]:
# ADR 0048 diagnostic: discover the real Kaggle notebook environment
# (Python/torch/CUDA, pre-installed packages, disk, GPU, no-internet check)
# before deciding how to package OLMo-2-1124-7B + Unsloth/bitsandbytes/peft offline.
import sys
import platform
import subprocess

print("python:", sys.version)
print("platform:", platform.platform())


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
import importlib

packages = [
    "torch", "transformers", "accelerate", "peft", "bitsandbytes",
    "unsloth", "unsloth_zoo", "trl", "xformers", "triton", "safetensors",
]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown")
        print(f"{pkg}: PRESENT, version={version}")
    except ImportError as exc:
        print(f"{pkg}: MISSING ({exc})")


In [ ]:
import torch
print("torch.__version__:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("total VRAM (GiB):", props.total_memory / (1024**3))
    print("device count:", torch.cuda.device_count())


In [ ]:
import shutil
for path in ["/kaggle/working", "/kaggle/temp", "/"]:
    try:
        total, used, free = shutil.disk_usage(path)
        print(f"{path}: total={total/1024**3:.1f}GiB used={used/1024**3:.1f}GiB free={free/1024**3:.1f}GiB")
    except FileNotFoundError:
        print(f"{path}: does not exist")


In [ ]:
import os
print("contents of /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "N/A")


In [ ]:
# Confirm internet is really disabled (expected to fail/timeout, that's success for this check)
import socket
socket.setdefaulttimeout(5)
try:
    import urllib.request
    urllib.request.urlopen("https://huggingface.co")
    print("INTERNET IS REACHABLE (unexpected, enable_internet should be false)")
except Exception as exc:
    print(f"internet unreachable as expected: {type(exc).__name__}: {exc}")
